# 13 — Cosmological $P(k)$ inference: `emcee` versus SBI

We now infer

$$
\theta=(\Omega_m,\sigma_8)
$$

from an idealized noisy linear matter power spectrum. The purpose
is a controlled method comparison, not a survey analysis.

**Fair-comparison rule:** `emcee` and NPE must see the same prior,
theory function, $k$ bins, observed realization, fixed covariance,
and summary transformation. If any of these differ, contour
agreement or disagreement is not interpretable.

Before running the notebook, predict:

1. which parameter mostly changes the power-spectrum amplitude;
2. which parameter also changes its shape; and
3. whether the posterior should be circular or correlated.

**Student work:** complete the symbolic-theory wrapper, covariance,
MCMC target, stochastic simulator, `sbi` NPE workflow, and
repeated-simulation coverage.

In [1]:
from pathlib import Path
import json
import os
import random
import sys
import time
import warnings

ROOT = Path(os.path.abspath('.'))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".mplconfig"))

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

SEED = 2605
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
warnings.filterwarnings("ignore", message="IProgress not found.*")

class NullTracker:
    '''Minimal no-op tracker that keeps classroom runs out of TensorBoard.'''

    @property
    def log_dir(self):
        return None

    def log_metric(self, name, value, step=None):
        pass

    def log_metrics(self, metrics, step=None):
        pass

    def log_params(self, params):
        pass

    def add_figure(self, name, figure, step=None):
        pass

    def flush(self):
        pass

COLORS = {
    "theta": "#7656A5",
    "data": "#2A9D8F",
    "observation": "#C94C4C",
    "learned": "#E6862E",
    "reference": "#2D6A9F",
    "gray": "#626C78",
}

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "figure.facecolor": "white",
    "axes.facecolor": "#FBFCFE",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10.5,
    "legend.frameon": False,
})

def savefig(fig, name):
    fig.text(
        0.995, 0.005, "analytic teaching model",
        ha="right", va="bottom", fontsize=7, color=COLORS["gray"],
    )
    path = OUTPUT_DIR / name
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"saved: {path.relative_to(ROOT)}")
    return path

def write_json(name, payload):
    path = OUTPUT_DIR / name
    with path.open("w") as stream:
        json.dump(payload, stream, indent=2, sort_keys=True)
        stream.write("\n")
    print(f"saved: {path.relative_to(ROOT)}")
    return path

In [5]:
import torch
import emcee
import corner
from matplotlib.lines import Line2D
import symbolic_pk_linear as wlin
from sbi.inference import NPE
from sbi.utils import BoxUniform

from cosmology_power import (
    FixedCosmology,
    covariance_cholesky,
    fixed_gaussian_covariance,
    linear_power_spectrum,
    linear_power_spectrum_batch,
    make_k_bins,
    mode_counts,
    simulate_power_spectrum,
    simulate_power_spectra,
    unwhiten,
    whiten,
)

torch.manual_seed(SEED)
torch.set_num_threads(1)

PARAMETER_NAMES = [r"$\Omega_m$", r"$\sigma_8$"]
PRIOR_LOW = np.array([0.20, 0.65])
PRIOR_HIGH = np.array([0.42, 0.97])
THETA_TRUE = np.array([0.315, 0.811])
FIXED = FixedCosmology(
    omega_b=0.049, h=0.674, n_s=0.965, redshift=0.0
)
VOLUME = 350.0**3  # (Mpc/h)^3
K_EDGES, K = make_k_bins(
    k_min_h_mpc=0.05,
    k_max_h_mpc=0.20,
    n_bins=15,
    spacing="log",
)

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


## 1. The symbolic linear matter power spectrum

We use the wide-prior symbolic approximation of Bartlett & Pandey:

$$
P_{\rm lin}(k,z)
=D^2(z)\,P_{\rm EH,nw}(k)\,
  \exp\!\left[\log F(k)\right].
$$

The package internally maps $\sigma_8$ to the required
normalization and recomputes the $\Omega_m$-dependent shape. We
call the published implementation directly rather than copying its
fitted coefficients into the notebook.

The tutorial remains inside the published ranges, but the stated
roughly $0.6\%$ linear-model RMS accuracy is still a theory
approximation. It should not be silently treated as exact in a
precision survey analysis.

In [ ]:
def student_power_spectrum(theta, k_h_mpc):
    '''Call the published wide-prior symbolic linear P(k) implementation.'''
    omega_m, sigma_8 = np.asarray(theta, dtype=float)

    # TODO 1: call wlin.symbolic_pklin in its documented order:
    # Omega_m, Omega_b, h, n_s, sigma_8, z, k. Return one P(k) vector.
    raise NotImplementedError

In [ ]:
power_from_student = student_power_spectrum(THETA_TRUE, K)
power_from_helper = linear_power_spectrum(
    THETA_TRUE[0], THETA_TRUE[1], K, fixed=FIXED
)
np.testing.assert_allclose(
    power_from_student, power_from_helper, rtol=1.0e-12
)

fig, axes = plt.subplots(
    1, 2, figsize=(11.5, 4.2), constrained_layout=True
)
for omega_m in [0.25, 0.315, 0.38]:
    power = student_power_spectrum([omega_m, 0.811], K)
    axes[0].loglog(K, power, lw=2, label=rf"$\Omega_m={omega_m}$")
for sigma_8 in [0.72, 0.811, 0.90]:
    power = student_power_spectrum([0.315, sigma_8], K)
    axes[1].loglog(K, power, lw=2, label=rf"$\sigma_8={sigma_8}$")
axes[0].set_title(r"Changing $\Omega_m$: amplitude and shape")
axes[1].set_title(r"Changing $\sigma_8$: normalization")
for ax in axes:
    ax.set(
        xlabel=r"$k\ [h\,{\rm Mpc}^{-1}]$",
        ylabel=r"$P_{\rm lin}(k)\ [({\rm Mpc}/h)^3]$",
    )
    ax.legend()
savefig(fig, "13_power_sensitivity.png")
plt.show()

## 2. A fixed Gaussian band-power covariance

For an isotropic shell,

$$
N_i=\frac{V}{6\pi^2}
\left(k_{i,\rm high}^3-k_{i,\rm low}^3\right),
\qquad
{\rm Var}[\hat P_i]
=\frac{2P_{\rm fid}(k_i)^2}{N_i}.
$$

Here $N_i$ follows the conventional count of Fourier wavevectors
in the shell, including the conjugate $\mathbf k,-\mathbf k$
pair; the factor of two in the covariance is consistent with that
convention.

We evaluate this covariance once at the fiducial cosmology and
keep it fixed. That makes the normalization term in the Gaussian
likelihood constant. The Gaussian band-power summary is a
controlled teaching approximation, not a positive-valued physical
field simulator. Starting at $k=0.05\,h\,{\rm Mpc}^{-1}$ keeps
every shell near or above 30 wavevectors and makes negative noisy
summaries rare. If $C$ depended on $\theta$, `emcee` would also
need $\log\det C(\theta)$ and the SBI simulator would need the same
parameter-dependent noise.

In [ ]:
def student_mode_counts(k_edges, volume):
    '''Continuum number of Fourier wavevectors in each shell.'''
    # TODO 2a: implement
    # N_i = V (k_high^3 - k_low^3) / (6 pi^2).
    raise NotImplementedError


def student_fixed_covariance(fiducial_power, counts):
    '''Fixed Gaussian matter-power covariance without shot noise.'''
    # TODO 2b: return the diagonal matrix with
    # Var[P_i] = 2 P_fid(k_i)^2 / N_i.
    raise NotImplementedError

In [ ]:
POWER_FIDUCIAL = linear_power_spectrum(
    THETA_TRUE[0], THETA_TRUE[1], K, fixed=FIXED
)
MODE_COUNTS = student_mode_counts(K_EDGES, VOLUME)
COVARIANCE = student_fixed_covariance(
    POWER_FIDUCIAL, MODE_COUNTS
)
CHOLESKY = covariance_cholesky(COVARIANCE)

np.testing.assert_allclose(
    MODE_COUNTS, mode_counts(K_EDGES, VOLUME)
)
np.testing.assert_allclose(
    COVARIANCE,
    fixed_gaussian_covariance(POWER_FIDUCIAL, MODE_COUNTS),
)
fractional_error = np.sqrt(np.diag(COVARIANCE)) / POWER_FIDUCIAL

fig, ax = plt.subplots(figsize=(8.5, 4.2), constrained_layout=True)
ax.loglog(K, MODE_COUNTS, "o-", color=COLORS["data"], label=r"$N_i$")
twin = ax.twinx()
twin.loglog(
    K, fractional_error, "s--",
    color=COLORS["observation"],
    label=r"$\sigma[P_i]/P_{\rm fid}$",
)
ax.set(
    xlabel=r"$k\ [h\,{\rm Mpc}^{-1}]$",
    ylabel="Fourier wavevector count",
    title="More high-k modes give smaller Gaussian errors",
)
twin.set_ylabel("fractional standard deviation")
handles = ax.lines + twin.lines
ax.legend(handles, [line.get_label() for line in handles])
savefig(fig, "13_mode_counts_and_errors.png")
plt.show()

## 3. One seeded observation and a useful transformation

We generate one observation,

$$
\hat{\mathbf P}_o
=\mathbf P(\theta_\ast)+L\boldsymbol\epsilon,
\qquad
\boldsymbol\epsilon\sim\mathcal N(\mathbf 0,I),
$$

and whiten every spectrum using the same fiducial mean and
Cholesky factor:

$$
\mathbf x=L^{-1}
\left(\hat{\mathbf P}-\mathbf P_{\rm fid}\right).
$$

This is an invertible rescaling, not a lossy compression. In these
coordinates the fixed covariance is the identity.

In [ ]:
OBSERVED_POWER = simulate_power_spectrum(
    THETA_TRUE, K, CHOLESKY, seed=SEED, fixed=FIXED
)
X_OBSERVED = whiten(
    OBSERVED_POWER, POWER_FIDUCIAL, CHOLESKY
)

fig, axes = plt.subplots(
    1, 2, figsize=(11.5, 4.1), constrained_layout=True
)
axes[0].errorbar(
    K, OBSERVED_POWER,
    yerr=np.sqrt(np.diag(COVARIANCE)),
    fmt="o", ms=4, color=COLORS["observation"],
    label="observed realization",
)
axes[0].plot(
    K, POWER_FIDUCIAL, color=COLORS["reference"], lw=2,
    label="fiducial mean",
)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set(
    xlabel=r"$k\ [h\,{\rm Mpc}^{-1}]$",
    ylabel=r"$P(k)\ [({\rm Mpc}/h)^3]$",
    title="Data space",
)
axes[0].legend()
axes[1].axhspan(-2, 2, color=COLORS["gray"], alpha=0.10)
axes[1].plot(K, X_OBSERVED, "o-", color=COLORS["data"])
axes[1].axhline(0, color=COLORS["gray"], lw=1)
axes[1].set_xscale("log")
axes[1].set(
    xlabel=r"$k\ [h\,{\rm Mpc}^{-1}]$",
    ylabel=r"$L^{-1}(\hat P-P_{\rm fid})$",
    title="Whitened summary",
)
savefig(fig, "13_observation_and_whitening.png")
plt.show()

## 4. Baseline: an explicit Gaussian likelihood with `emcee`

In whitened coordinates,

$$
\log p(\mathbf x_o\mid\theta)
=-\frac12
\left\|
\mathbf x_o-L^{-1}
[\mathbf P(\theta)-\mathbf P_{\rm fid}]
\right\|^2+\text{constant}.
$$

`emcee` evaluates this target repeatedly. The simulator used by
SBI below draws noise from the same distribution instead of
evaluating this expression.

In [ ]:
def log_prior(theta):
    '''Uniform prior with explicit zero density outside its rectangle.'''
    theta = np.asarray(theta, dtype=float)
    # TODO 3a: return 0 inside [PRIOR_LOW, PRIOR_HIGH] and
    # -np.inf outside. Both endpoints may be included.
    raise NotImplementedError


def log_likelihood(theta):
    '''Fixed-covariance Gaussian log likelihood in whitened coordinates.'''
    if not np.isfinite(log_prior(theta)):
        return -np.inf

    # TODO 3b: evaluate P(k|theta) with fixed=FIXED, whiten it
    # with the *fixed* fiducial mean and Cholesky factor, then return
    # -0.5 * ||X_OBSERVED - model_white||^2.
    raise NotImplementedError


def log_posterior(theta):
    prior_value = log_prior(theta)
    if not np.isfinite(prior_value):
        return -np.inf
    return prior_value + log_likelihood(theta)

In [ ]:
n_walkers = 32
n_steps = 2_600
proposal_rng = np.random.default_rng(SEED + 1)
initial = proposal_rng.uniform(
    low=PRIOR_LOW,
    high=PRIOR_HIGH,
    size=(n_walkers, 2),
)

np.random.seed(SEED + 2)  # emcee's internal random state
mcmc = emcee.EnsembleSampler(n_walkers, 2, log_posterior)
started = time.perf_counter()
mcmc.run_mcmc(initial, n_steps, progress=False)
mcmc_seconds = time.perf_counter() - started

chain = mcmc.get_chain()
emcee_samples = mcmc.get_chain(
    discard=700, thin=4, flat=True
)
acceptance = float(np.mean(mcmc.acceptance_fraction))
autocorrelation_time = mcmc.get_autocorr_time(discard=700)
steps_per_tau = (n_steps - 700) / autocorrelation_time
effective_samples = (
    n_walkers * (n_steps - 700) / autocorrelation_time
)
print(
    f"emcee: {emcee_samples.shape[0]} retained samples, "
    f"acceptance={acceptance:.3f}, time={mcmc_seconds:.1f} s"
)
print(
    "rough integrated autocorrelation time:",
    np.round(autocorrelation_time, 1),
)
print("post-burn-in steps per tau:", np.round(steps_per_tau, 1))
print(
    "rough effective samples after burn-in:",
    np.round(effective_samples).astype(int),
)
if not 0.15 < acceptance < 0.75:
    raise RuntimeError("unexpected emcee acceptance fraction")

fig, axes = plt.subplots(
    2, 1, figsize=(9.5, 5.3), sharex=True,
    constrained_layout=True,
)
for parameter, ax, truth, color in zip(
    range(2), axes, THETA_TRUE,
    [COLORS["theta"], COLORS["data"]],
):
    ax.plot(chain[:, :, parameter], color=color, alpha=0.18, lw=0.6)
    ax.axhline(truth, color=COLORS["observation"], lw=1.5)
    ax.set_ylabel(PARAMETER_NAMES[parameter])
axes[-1].set_xlabel("MCMC step")
fig.suptitle("Trace inspection precedes contour comparison")
savefig(fig, "13_emcee_trace.png")
plt.show()

## 5. SBI: learn $p(\theta\mid\mathbf x)$ from simulations

For NPE we generate joint pairs,

$$
\theta_i\sim p(\theta),\qquad
\hat{\mathbf P}_i
=\mathbf P(\theta_i)+L\boldsymbol\epsilon_i,
$$

whiten them exactly as the observation, and fit

$$
q_\phi(\theta\mid\mathbf x)
\approx p(\theta\mid\mathbf x).
$$

The 12,000 simulations cover the full prior, so this estimator is
amortized: after training, it can answer many observation queries.

In [ ]:
def simulate_whitened_summaries(theta, seed):
    '''Simulate noisy P(k) values and return their whitened summaries.'''
    theta = np.asarray(theta, dtype=float)

    # TODO 4: call simulate_power_spectra with K, CHOLESKY, and
    # fixed=FIXED, then whiten every simulated spectrum relative to
    # POWER_FIDUCIAL. The result has shape (n_simulations, n_k_bins).
    raise NotImplementedError

In [ ]:
sbi_prior = BoxUniform(
    low=torch.as_tensor(PRIOR_LOW, dtype=torch.float32),
    high=torch.as_tensor(PRIOR_HIGH, dtype=torch.float32),
)
n_sbi_simulations = 12_000
theta_sbi = sbi_prior.sample((n_sbi_simulations,))
x_sbi_numpy = simulate_whitened_summaries(
    theta_sbi.numpy(), seed=SEED + 3
)
x_sbi = torch.as_tensor(x_sbi_numpy, dtype=torch.float32)
training_power = unwhiten(
    x_sbi_numpy, POWER_FIDUCIAL, CHOLESKY
)
negative_training_fraction = float(
    np.mean(np.any(training_power <= 0.0, axis=1))
)

assert theta_sbi.shape == (n_sbi_simulations, 2)
assert x_sbi.shape == (n_sbi_simulations, K.size)
assert torch.isfinite(x_sbi).all()
print("training pairs:", tuple(theta_sbi.shape), tuple(x_sbi.shape))
print(
    "rare negative Gaussian-summary rows:",
    f"{negative_training_fraction:.3%}",
)
if negative_training_fraction >= 0.005:
    raise RuntimeError(
        "Gaussian summary approximation produces too many "
        "negative band powers"
    )

In [ ]:
# TODO 5: reproduce the four NPE stages:
# 1. create NPE(..., tracker=NullTracker()) with a MAF density estimator;
# 2. append (theta_sbi, x_sbi);
# 3. train with batch size 256, at most 100 epochs, and early stopping after
#    15 validation epochs;
# 4. build the posterior object.
raise NotImplementedError

In [ ]:
x_observed_tensor = torch.as_tensor(
    X_OBSERVED, dtype=torch.float32
)
sbi_samples = (
    sbi_posterior.sample(
        (8_000,),
        x=x_observed_tensor,
        show_progress_bars=False,
    )
    .cpu().numpy()
)
assert sbi_samples.shape == (8_000, 2)
assert np.all(
    (sbi_samples >= PRIOR_LOW) & (sbi_samples <= PRIOR_HIGH)
)

Because this teaching likelihood is explicitly available, we can
also integrate it on a dense grid. That numerical grid is an
independent reference for both samplers; it prevents two
approximate methods from appearing to agree merely because they
share a bias.

In [ ]:
omega_grid = np.linspace(PRIOR_LOW[0], PRIOR_HIGH[0], 181)
sigma8_grid = np.linspace(PRIOR_LOW[1], PRIOR_HIGH[1], 181)
omega_mesh, sigma8_mesh = np.meshgrid(
    omega_grid, sigma8_grid, indexing="ij"
)
grid_theta = np.column_stack([
    omega_mesh.ravel(), sigma8_mesh.ravel()
])
grid_power = linear_power_spectrum_batch(
    grid_theta, K, fixed=FIXED
)
grid_white = whiten(
    grid_power, POWER_FIDUCIAL, CHOLESKY
)
grid_log_probability = -0.5 * np.sum(
    (X_OBSERVED[None, :] - grid_white) ** 2,
    axis=1,
)
grid_log_probability -= np.max(grid_log_probability)
grid_probability = np.exp(grid_log_probability)
grid_probability /= np.sum(grid_probability)

grid_mean = np.sum(
    grid_probability[:, None] * grid_theta, axis=0
)
grid_delta = grid_theta - grid_mean
grid_covariance = np.einsum(
    "n,ni,nj->ij",
    grid_probability,
    grid_delta,
    grid_delta,
)
grid_std = np.sqrt(np.diag(grid_covariance))
grid_correlation = (
    grid_covariance[0, 1] / (grid_std[0] * grid_std[1])
)
print("grid reference mean:", np.round(grid_mean, 6))
print("grid reference std:", np.round(grid_std, 6))
print("grid reference correlation:", f"{grid_correlation:.3f}")

## 6. Compare the posteriors, not just their means

The MCMC and NPE contours should agree with the numerical grid
because the likelihood is Gaussian and the SBI simulator uses
precisely the same stochastic model. Disagreement would point to
finite-chain error, finite simulation/training error, estimator
error, or an implementation mismatch—not to a new cosmological
effect.

In [ ]:
contour_range = [
    (PRIOR_LOW[0], PRIOR_HIGH[0]),
    (PRIOR_LOW[1], PRIOR_HIGH[1]),
]
figure = corner.corner(
    emcee_samples,
    labels=PARAMETER_NAMES,
    truths=THETA_TRUE,
    truth_color="#111111",
    range=contour_range,
    bins=45,
    color=COLORS["reference"],
    plot_datapoints=False,
    fill_contours=False,
    levels=(0.68, 0.95),
    smooth=1.0,
    hist_kwargs={"density": True},
)
corner.corner(
    sbi_samples,
    fig=figure,
    range=contour_range,
    bins=45,
    color=COLORS["learned"],
    plot_datapoints=False,
    fill_contours=False,
    levels=(0.68, 0.95),
    smooth=1.0,
    hist_kwargs={"density": True},
)
figure.legend(
    handles=[
        Line2D([0], [0], color=COLORS["reference"], lw=2, label="emcee"),
        Line2D([0], [0], color=COLORS["learned"], lw=2, label="NPE"),
        Line2D([0], [0], color="#111111", lw=1.4, label="truth"),
    ],
    loc="upper right", bbox_to_anchor=(0.96, 0.96),
)
figure.suptitle(
    "emcee and NPE give consistent contours for one observation",
    y=1.02,
)
savefig(figure, "13_emcee_vs_sbi_contours.png")
plt.show()

emcee_mean = emcee_samples.mean(axis=0)
sbi_mean = sbi_samples.mean(axis=0)
emcee_std = emcee_samples.std(axis=0)
sbi_std = sbi_samples.std(axis=0)
emcee_covariance = np.cov(emcee_samples, rowvar=False)
sbi_covariance = np.cov(sbi_samples, rowvar=False)
normalized_mean_shift = np.abs(emcee_mean - sbi_mean) / grid_std
std_ratio = sbi_std / emcee_std
emcee_to_grid_std_ratio = emcee_std / grid_std
sbi_to_grid_std_ratio = sbi_std / grid_std
emcee_to_grid_area_ratio = np.sqrt(
    np.linalg.det(emcee_covariance)
    / np.linalg.det(grid_covariance)
)
sbi_to_grid_area_ratio = np.sqrt(
    np.linalg.det(sbi_covariance)
    / np.linalg.det(grid_covariance)
)
comparison = {
    name: {
        "grid_mean": float(grid_mean[index]),
        "emcee_mean": float(emcee_mean[index]),
        "sbi_mean": float(sbi_mean[index]),
        "grid_std": float(grid_std[index]),
        "emcee_std": float(emcee_std[index]),
        "sbi_std": float(sbi_std[index]),
        "emcee_grid_mean_shift_in_sigma": float(
            abs(emcee_mean[index] - grid_mean[index])
            / grid_std[index]
        ),
        "sbi_grid_mean_shift_in_sigma": float(
            abs(sbi_mean[index] - grid_mean[index])
            / grid_std[index]
        ),
        "emcee_sbi_mean_shift_in_grid_sigma": float(
            normalized_mean_shift[index]
        ),
        "sbi_to_emcee_std_ratio": float(std_ratio[index]),
        "emcee_to_grid_std_ratio": float(
            emcee_to_grid_std_ratio[index]
        ),
        "sbi_to_grid_std_ratio": float(
            sbi_to_grid_std_ratio[index]
        ),
    }
    for index, name in enumerate(["Omega_m", "sigma_8"])
}
comparison["joint"] = {
    "grid_correlation": float(grid_correlation),
    "emcee_correlation": float(
        np.corrcoef(emcee_samples.T)[0, 1]
    ),
    "sbi_correlation": float(
        np.corrcoef(sbi_samples.T)[0, 1]
    ),
    "emcee_to_grid_ellipse_area_ratio": float(
        emcee_to_grid_area_ratio
    ),
    "sbi_to_grid_ellipse_area_ratio": float(
        sbi_to_grid_area_ratio
    ),
}
print(json.dumps(comparison, indent=2))

Cross-method agreement for this one observation is useful but is
not a calibration test. We therefore add two checks with different
targets.

## 7. Posterior predictive check

Draw parameters from the NPE posterior, simulate fresh noisy
spectra. The plotted interval is pointwise, so it does not by
itself establish joint agreement across all bins. We therefore
also compare a joint whitened-residual discrepancy for the
observation and each replicated spectrum. Passing still does not
prove that the posterior has correct coverage.

In [ ]:
ppc_indices = np.random.default_rng(SEED + 4).choice(
    sbi_samples.shape[0], size=400, replace=False
)
ppc_power = simulate_power_spectra(
    sbi_samples[ppc_indices],
    K,
    CHOLESKY,
    seed=SEED + 5,
    fixed=FIXED,
)
ppc_mean_power = linear_power_spectrum_batch(
    sbi_samples[ppc_indices], K, fixed=FIXED
)
replicated_residual = whiten(
    ppc_power, ppc_mean_power, CHOLESKY
)
observed_residual = whiten(
    np.broadcast_to(OBSERVED_POWER, ppc_mean_power.shape),
    ppc_mean_power,
    CHOLESKY,
)
replicated_chi2 = np.sum(replicated_residual**2, axis=1)
observed_chi2 = np.sum(observed_residual**2, axis=1)
ppc_tail_probability = float(
    np.mean(replicated_chi2 >= observed_chi2)
)
ppc_low, ppc_median, ppc_high = np.quantile(
    ppc_power, [0.16, 0.50, 0.84], axis=0
)

fig, ax = plt.subplots(figsize=(8.8, 4.5), constrained_layout=True)
ax.fill_between(
    K, ppc_low, ppc_high,
    color=COLORS["learned"], alpha=0.30,
    label="NPE posterior predictive 68%",
)
ax.plot(K, ppc_median, color=COLORS["learned"], lw=2)
ax.plot(
    K, OBSERVED_POWER, "o",
    color=COLORS["observation"], ms=4, label="observation",
)
ax.plot(
    K, POWER_FIDUCIAL,
    color=COLORS["reference"], ls="--", label="fiducial mean",
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(
    xlabel=r"$k\ [h\,{\rm Mpc}^{-1}]$",
    ylabel=r"$P(k)\ [({\rm Mpc}/h)^3]$",
    title=(
        "Pointwise predictive band; joint residual tail "
        f"probability = {ppc_tail_probability:.2f}"
    ),
)
ax.legend()
savefig(fig, "13_posterior_predictive.png")
plt.show()

## 8. Expected marginal coverage

Repeat the whole observation-generating step:

$$
\theta^\ast\sim p(\theta),\qquad
\mathbf x^\ast\sim p(\mathbf x\mid\theta^\ast),
$$

query the already-trained NPE, and check how often each true
parameter lies inside its equal-tailed interval. This tests global
marginal calibration over the training prior. It can still miss
incorrect posterior correlations, and 80 test cases still give
visible binomial uncertainty.

In [ ]:
def marginal_coverage_2d(posterior_samples, theta_true, probability):
    '''Equal-tailed coverage for each parameter in a batch of posteriors.'''
    # TODO 6 (8 min): compute a central interval along the posterior-draw
    # axis and return the fraction of true values inside for each parameter.
    raise NotImplementedError

In [ ]:
n_coverage_cases = 80
theta_coverage = sbi_prior.sample((n_coverage_cases,))
x_coverage = simulate_whitened_summaries(
    theta_coverage.numpy(), seed=SEED + 6
)
coverage_draws = []
for x_case in x_coverage:
    draws = (
        sbi_posterior.sample(
            (400,),
            x=torch.as_tensor(x_case, dtype=torch.float32),
            show_progress_bars=False,
        )
        .cpu().numpy()
    )
    coverage_draws.append(draws)
coverage_draws = np.stack(coverage_draws)
theta_coverage_numpy = theta_coverage.numpy()

levels = np.array([0.50, 0.68, 0.80, 0.95])
coverage_values = np.stack([
    marginal_coverage_2d(
        coverage_draws, theta_coverage_numpy, level
    )
    for level in levels
])
binomial_error = np.sqrt(
    levels * (1.0 - levels) / n_coverage_cases
)
coverage_z = (
    coverage_values - levels[:, None]
) / binomial_error[:, None]
print("marginal coverage:")
print(np.round(coverage_values, 4))
print("deviation from nominal [binomial standard errors]:")
print(np.round(coverage_z, 2))

fig, ax = plt.subplots(figsize=(6.0, 5.1), constrained_layout=True)
for index, (name, color, marker) in enumerate([
    (r"$\Omega_m$", COLORS["theta"], "o"),
    (r"$\sigma_8$", COLORS["data"], "s"),
]):
    ax.errorbar(
        levels, coverage_values[:, index],
        yerr=binomial_error, fmt=marker,
        color=color, capsize=4, label=name,
    )
ax.plot([0.4, 1.0], [0.4, 1.0], "--", color=COLORS["gray"])
ax.set(
    xlim=(0.45, 0.98), ylim=(0.40, 1.0),
    xlabel="nominal interval probability",
    ylabel="empirical marginal coverage",
    title="Independent simulations test amortized calibration",
)
ax.legend()
savefig(fig, "13_coverage.png")
plt.show()

np.savez(
    OUTPUT_DIR / "13_posterior_samples.npz",
    k_h_mpc=K,
    theta_true=THETA_TRUE,
    observed_power=OBSERVED_POWER,
    covariance=COVARIANCE,
    emcee_samples=emcee_samples,
    sbi_samples=sbi_samples,
)
write_json(
    "13_inference_summary.json",
    {
        "seed": SEED,
        "truth": THETA_TRUE.tolist(),
        "fixed_parameters": {
            "Omega_b": FIXED.omega_b,
            "h": FIXED.h,
            "n_s": FIXED.n_s,
            "z": FIXED.redshift,
        },
        "prior_low": PRIOR_LOW.tolist(),
        "prior_high": PRIOR_HIGH.tolist(),
        "volume_mpc_h3": VOLUME,
        "k_range_h_mpc": [float(K_EDGES[0]), float(K_EDGES[-1])],
        "n_k_bins": int(K.size),
        "n_sbi_simulations": n_sbi_simulations,
        "n_coverage_cases": n_coverage_cases,
        "negative_training_summary_fraction": (
            negative_training_fraction
        ),
        "emcee_acceptance_fraction": acceptance,
        "emcee_autocorrelation_time": autocorrelation_time.tolist(),
        "emcee_steps_per_tau": steps_per_tau.tolist(),
        "emcee_effective_samples": effective_samples.tolist(),
        "mcmc_seconds": mcmc_seconds,
        "grid_mean": grid_mean.tolist(),
        "grid_covariance": grid_covariance.tolist(),
        "posterior_predictive_joint_tail_probability": (
            ppc_tail_probability
        ),
        "comparison": comparison,
        "coverage_levels": levels.tolist(),
        "marginal_coverage": coverage_values.tolist(),
    },
)

## Interpretation and limitations

In this controlled Gaussian-summary problem, `emcee`, NPE, and the
dense likelihood grid should agree because they represent the same
posterior in three different ways:

- `emcee` repeatedly evaluates an available Gaussian likelihood;
- NPE learns the posterior from prior-predictive simulations and
  then samples it directly.

The repeated-simulation coverage plot is the more demanding test.
With only 80 cases its error bars are still broad, and the saved
run places the $\Omega_m$ coverage about two to two-and-a-half
binomial standard errors below nominal across the tested levels.
Treat that pattern as a calibration warning to investigate with
more held-out simulations—not as evidence of perfect coverage.

The NPE training cost is amortized across future observations. For
a single cheap Gaussian likelihood, MCMC is usually the simpler
tool. SBI becomes compelling when the forward model remains
simulatable but the likelihood is unavailable, biased by a
Gaussian approximation, or prohibitively expensive to evaluate.

The generated vector is a Gaussian approximation to band-power
estimates, not a realization of a positive matter density field.
Rare negative noisy entries are an admitted limitation of that
approximation and are monitored above. The notebook deliberately
omits nonlinear evolution, survey
windows, masks, biased tracers, RSD, nuisance parameters,
non-Gaussian covariance, and emulator uncertainty. Adding any of
them changes the scientific model and requires new validation.

**Optional extension.** Make the covariance depend on
$P(k\mid\theta)$. Before coding, write down both changes required:
the MCMC log determinant and the parameter-dependent simulator
noise.

**Sources**

- [Bartlett & Pandey (2025), symbolic matter-power-spectrum
  expressions](https://arxiv.org/abs/2510.18749)
- [`symbolic_pofk` source repository](https://github.com/DeaglanBartlett/symbolic_pofk)
- [Foreman-Mackey et al. (2013), `emcee`](https://arxiv.org/abs/1202.3665)
- [`sbi` method-selection guide](https://sbi.readthedocs.io/en/latest/how_to_guide/06_choosing_inference_method.html)
- [`sbi` diagnostic guide](https://sbi.readthedocs.io/en/latest/how_to_guide/14_choose_diagnostic_tool.html)
- [A Practical Guide to Simulation-Based Inference](https://arxiv.org/abs/2508.12939)